# 关键字检索（Keyword Search / BM25）

## 用途
教学演示 - 基于关键词的传统检索方法

## 核心概念
- BM25 算法原理
- 关键词匹配 vs 语义匹配
- 倒排索引

## 运行前检查

1. 已安装依赖：`pip install -r ../requirements.txt`
2. 已完成 `02_vector_search.py` 的学习

In [1]:
from rag_examples.milvus_config import MILVUS_URI
from dotenv import load_dotenv
load_dotenv()

True

# 第一部分：理解关键字检索

## 什么是关键字检索？

### 定义
关键字检索 = 基于关键词匹配的检索，查找包含查询词的文档，不考虑语义

### 💡 生活化比喻
关键字检索 = "查字典"

- 查"苹果"→ 找包含"苹果"这两个字的页面
- 不会找到"iPhone"、"水果"相关内容（除非也包含"苹果"这个词）

### 📦 BM25 算法原理

BM25 (Best Matching 25) 是最经典的关键字检索算法

**核心思想：**

1. **TF（词频 Term Frequency）**
   - 词在文档中出现越多，越重要
   - 但要避免堆砌（边际效应递减）

2. **IDF（逆文档频率 Inverse Document Frequency）**
   - 在越少文档中出现的词，越重要
   - "的"、"是"等常用词 IDF 低
   - "量子计算"等专有词 IDF 高

3. **文档长度归一化**
   - 长文档词频天然高，需要"打折"
   - 避免长文档总是排在前面

**公式（简化版）:**
`Score(Q, D) = Σ [IDF(qi) × TF(qi, D)]`

### 🔑 关键字检索 vs 向量检索

| 维度 | 关键字检索 | 向量检索 |
|------|------------|----------|
| 原理 | 词匹配 | 语义匹配 |
| 优点 | 精确、可解释 | 理解语义 |
| 缺点 | 无法理解同义词 | 可能过度泛化 |
| 适用 | 专有名词、精确查询 | 语义理解、模糊查询 |

### ⚠️ 应用场景

1. 精确匹配：产品型号 "iPhone 15 Pro Max"
2. 专有名词：人名、地名、品牌名
3. 代码检索：函数名、变量名
4. 法律/医疗：精确术语匹配

In [2]:
def simple_keyword_match():
    """
    简单的关键词匹配演示
    """
    print("=" * 60)
    print("示例 1: 简单关键词匹配")
    print("=" * 60)

    # 测试文档
    documents = [
        "人工智能是模拟人类智能的计算机科学领域。",
        "机器学习通过训练数据让计算机自动学习规律。",
        "深度学习是机器学习的子集，使用神经网络。",
        "自然语言处理让计算机理解人类语言。",
        "计算机视觉让计算机看懂图像和视频。",
    ]

    # 查询
    query = "机器学习"

    print(f"查询：{query}")
    print(f"文档数量：{len(documents)}\n")

    # 简单匹配：文档中包含查询词
    matched = []
    for i, doc in enumerate(documents):
        if query in doc:
            matched.append((i, doc))

    print("匹配结果（包含关键词的文档）：")
    for i, doc in matched:
        print(f"  [{i+1}] {doc}")

    print("\n💡 问题：")
    print("  第 3 句'深度学习是机器学习的子集'也包含'机器学习'")
    print("  但如果是'AI 学习'、'训练模型'等同义词就匹配不到了")

In [3]:
simple_keyword_match()

示例 1: 简单关键词匹配
查询：机器学习
文档数量：5

匹配结果（包含关键词的文档）：
  [2] 机器学习通过训练数据让计算机自动学习规律。
  [3] 深度学习是机器学习的子集，使用神经网络。

💡 问题：
  第 3 句'深度学习是机器学习的子集'也包含'机器学习'
  但如果是'AI 学习'、'训练模型'等同义词就匹配不到了


# 第二部分：BM25 算法实现

In [ ]:
class SimpleBM25:
    """
    简化的 BM25 实现（用于教学演示）
    """

    def __init__(self, documents, k1=1.5, b=0.75):
        """
        参数:
            documents: 文档列表
            k1: 词频饱和度参数（默认 1.5）
            b: 长度归一化参数（默认 0.75）
        """
        self.documents = documents
        self.k1 = k1
        self.b = b

        # 预处理：分词（中文用简单字符分割）
        self.tokenized_docs = []
        for doc in documents:
            # 简单分词：按字符分割（实际应该用 jieba 等分词工具）
            tokens = list(doc.lower())
            self.tokenized_docs.append(tokens)

        # 计算文档频率（DF）
        self.df = {}
        for tokens in self.tokenized_docs:
            unique_tokens = set(tokens)
            for token in unique_tokens:
                self.df[token] = self.df.get(token, 0) + 1

        self.num_docs = len(documents)
        self.avg_doc_len = sum(len(t) for t in self.tokenized_docs) / self.num_docs

    def _idf(self, term):
        """计算 IDF"""
        df = self.df.get(term, 0)
        # IDF 公式：log((N - df + 0.5) / (df + 0.5) + 1)
        import math
        return math.log((self.num_docs - df + 0.5) / (df + 0.5) + 1)

    def _tf(self, term, doc_index):
        """计算 TF"""
        tokens = self.tokenized_docs[doc_index]
        freq = tokens.count(term)
        doc_len = len(tokens)

        # BM25 TF 公式
        numerator = freq * (self.k1 + 1)
        denominator = freq + self.k1 * (1 - self.b + self.b * doc_len / self.avg_doc_len)
        return numerator / denominator

    def search(self, query, top_k=3):
        """
        搜索与查询最相关的文档

        参数:
            query: 查询字符串
            top_k: 返回 Top-K 结果
        返回:
            (doc_index, score, doc_content) 列表
        """
        import math

        # 查询分词
        query_terms = list(query.lower())

        # 计算每个文档的得分
        scores = []
        for i in range(len(self.documents)):
            score = 0
            for term in query_terms:
                idf = self._idf(term)
                tf = self._tf(term, i)
                score += idf * tf
            scores.append((i, score, self.documents[i]))

        # 按分数排序
        scores.sort(key=lambda x: x[1], reverse=True)

        return scores[:top_k]

In [ ]:
def demo_bm25_search():
    """
    演示 BM25 搜索
    """
    print()
    print("=" * 60)
    print("示例 2: BM25 算法演示")
    print("=" * 60)

    # 测试文档
    documents = [
        "人工智能是模拟人类智能的计算机科学领域，包括机器学习。",
        "机器学习通过训练数据让计算机自动学习规律和模式。",
        "深度学习是机器学习的子集，使用多层神经网络。",
        "自然语言处理是 AI 的重要分支，研究语言理解。",
        "计算机视觉让计算机能够看懂图像和视频内容。",
        "推荐系统根据用户历史行为推荐相关内容。",
    ]

    print("文档库：")
    for i, doc in enumerate(documents):
        print(f"  [{i+1}] {doc}")
    print()

    # 创建 BM25 索引
    bm25 = SimpleBM25(documents)

    # 测试查询
    queries = ["机器学习", "人工智能", "深度学习"]

    for query in queries:
        print(f"查询：'{query}'")
        print("-" * 50)

        results = bm25.search(query, top_k=3)

        for i, (doc_idx, score, doc) in enumerate(results):
            print(f"  [{i+1}] 分数：{score:.4f}")
            print(f"      内容：{doc}")
        print()

In [ ]:
demo_bm25_search()

# 第三部分：使用 rank-bm25 库

In [4]:
def bm25_with_library():
    """
    使用 rank-bm25 库进行 BM25 检索
    """
    print("=" * 60)
    print("示例 3: 使用 rank-bm25 库")
    print("=" * 60)

    try:
        from rank_bm25 import BM25Okapi
        import jieba

        # 测试文档
        documents = [
            "人工智能是模拟人类智能的计算机科学领域，包括机器学习、深度学习等技术。",
            "机器学习通过训练数据让计算机自动学习规律，无需显式编程。",
            "深度学习是机器学习的子集，使用多层神经网络模拟人脑。",
            "自然语言处理让计算机理解和生成人类语言，应用广泛。",
            "计算机视觉让计算机能够看懂图像和视频，用于人脸识别。",
        ]

        print("文档库：")
        for i, doc in enumerate(documents):
            print(f"  [{i+1}] {doc}")
        print()

        # 中文分词
        print("正在进行中文分词...")
        tokenized_docs = [list(jieba.cut(doc)) for doc in documents]
        print("✓ 分词完成\n")

        # 创建 BM25 索引
        bm25 = BM25Okapi(tokenized_docs)

        # 测试查询
        queries = [
            "机器学习是什么",
            "深度学习和神经网络",
            "自然语言处理应用"
        ]

        for query in queries:
            print(f"查询：'{query}'")
            print("-" * 50)

            # 查询分词
            tokenized_query = list(jieba.cut(query))

            # 获取 BM25 分数
            scores = bm25.get_scores(tokenized_query)

            # 排序
            ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)

            # 显示 Top-3
            for i, (doc_idx, score) in enumerate(ranked[:3]):
                print(f"  [{i+1}] 分数：{score:.4f}")
                print(f"      内容：{documents[doc_idx][:50]}...")
            print()

    except ImportError as e:
        print(f"需要安装：pip install rank-bm25 jieba")
        print(f"错误：{e}")

In [5]:
bm25_with_library()

示例 3: 使用 rank-bm25 库
需要安装：pip install rank-bm25 jieba
错误：No module named 'rank_bm25'


# 第四部分：关键字检索 vs 向量检索

In [ ]:
def keyword_vs_vector_search():
    """
    对比关键字检索和向量检索的效果
    """
    print("=" * 60)
    print("示例 4: 关键字检索 vs 向量检索")
    print("=" * 60)

    # 测试文档（设计成能体现差异的情况）
    documents = [
        "机器学习是人工智能的核心技术，通过数据训练模型。",  # 包含"机器学习"
        "深度学习使用神经网络，是 ML 的重要分支。",  # 包含"ML"缩写
        "AI 模型通过训练可以自动学习规律和模式。",  # 同义词
        "神经网络的训练需要大量数据和计算资源。",  # 相关但不含关键词
    ]

    print("文档库：")
    for i, doc in enumerate(documents):
        print(f"  [{i+1}] {doc}")
    print()

    # 查询："AI 学习"（与"机器学习"语义相近，但字面不同）
    query = "AI 学习"
    print(f"查询：'{query}'")
    print()

    # 关键字检索结果
    print("1. 关键字检索结果：")
    print("   （查找包含'A'、'I'、'学'、'习'字符的文档）")
    matched_kw = [doc for doc in documents if 'A' in doc or 'I' in doc or '学习' in doc]
    for i, doc in enumerate(matched_kw):
        print(f"   - {doc[:40]}...")

    print()

    # 向量检索结果（模拟）
    print("2. 向量检索结果（模拟）：")
    print("   （理解'AI 学习'的语义，找到相关文档）")
    # 模拟：假设向量检索能理解语义
    vector_results = [
        (0, 0.85),  # 第 1 句最相关（AI、学习都有）
        (2, 0.78),  # 第 3 句相关（AI、学习）
        (1, 0.65),  # 第 2 句相关（神经网络）
    ]
    for doc_idx, score in vector_results:
        print(f"   [{score:.2f}] {documents[doc_idx][:40]}...")

    print()
    print("💡 结论：")
    print("   关键字检索：精确匹配字面，无法理解同义词")
    print("   向量检索：理解语义，能找到相关但字面不同的内容")
    print("   → 混合检索结合两者优势")

In [ ]:
keyword_vs_vector_search()

# 第五部分：BM25 参数调优

In [ ]:
def bm25_parameter_tuning():
    """
    BM25 参数调优演示
    """
    print()
    print("=" * 60)
    print("示例 5: BM25 参数调优")
    print("=" * 60)

    documents = [
        "机器学习是人工智能的核心技术。",
        "机器学习通过数据训练模型，机器学习应用广泛。",  # 长文档，"机器学习"出现 2 次
        "深度学习也是 ML 的一种形式。",
    ]

    query = "机器学习"

    print(f"查询：'{query}'")
    print("文档库：")
    for i, doc in enumerate(documents):
        print(f"  [{i+1}] {doc} ({len(doc)}字)")
    print()

    # 不同 k1 参数对比
    print("不同 k1 值的影响（b=0.75 固定）：")
    print("-" * 50)

    for k1 in [0.5, 1.2, 2.0]:
        bm25 = SimpleBM25(documents, k1=k1, b=0.75)
        results = bm25.search(query, top_k=3)

        print(f"\nk1={k1}:")
        for doc_idx, score, _ in results:
            print(f"  文档{doc_idx+1}: {score:.4f}")

    print("\n💡 参数说明：")
    print("   k1: 控制词频饱和度")
    print("       - k1 小：词频影响小，几次出现后分数不再显著增加")
    print("       - k1 大：词频影响大，多次出现会持续增加分数")
    print("       - 推荐值：1.2-2.0")
    print()
    print("   b: 控制长度归一化")
    print("       - b=0: 不考虑文档长度")
    print("       - b=1: 完全归一化")
    print("       - 推荐值：0.5-0.8")

In [ ]:
bm25_parameter_tuning()

# 第六部分：关键字检索最佳实践

In [ ]:
def keyword_search_best_practices():
    """
    关键字检索最佳实践
    """
    print()
    print("=" * 60)
    print("示例 6: 关键字检索最佳实践")
    print("=" * 60)

    print("""
┌─────────────────────────────────────────────────────────┐
│ 关键字检索最佳实践                                      │
├─────────────────────────────────────────────────────────┤
│                                                         │
│ 1. 适用场景                                             │
│    ✓ 专有名词检索（人名、地名、品牌）                  │
│    ✓ 代码/技术术语检索                                   │
│    ✓ 精确匹配需求                                        │
│    ✗ 语义理解需求（应用向量检索）                      │
│                                                         │
│ 2. 中文分词选择                                         │
│    - jieba: 轻量级，适合通用场景                        │
│    - HanLP: 功能丰富，支持实体识别                      │
│    - THULAC: 清华出品，学术场景                         │
│                                                         │
│ 3. 优化技巧                                             │
│    - 同义词扩展：查询"AI"时同时查"人工智能"             │
│    - 词干提取：英文场景下还原词根                        │
│    - 停用词过滤：去除"的"、"是"等无用词                 │
│                                                         │
│ 4. BM25 参数建议                                         │
│    - k1: 1.2-2.0（默认 1.5）                            │
│    - b: 0.5-0.8（默认 0.75）                            │
│                                                         │
└─────────────────────────────────────────────────────────┘

💡 混合检索策略：

在 RAG 系统中，推荐结合关键字检索和向量检索：

```python
# 1. 分别检索
keyword_results = bm25.search(query, top_k=10)
vector_results = vector_search(query, top_k=10)

# 2. 融合结果（RRF 倒数排名融合）
final_results = reciprocal_rank_fusion(
    keyword_results,
    vector_results,
    k=60
)

# 3. 返回 Top-5
return final_results[:5]
```
""")

In [ ]:
keyword_search_best_practices()

# 学习总结

## 关键概念

| 概念 | 说明 |
|------|------|
| BM25 | 最经典的关键字检索算法 |
| TF | 词频，词在文档中出现越多越重要 |
| IDF | 逆文档频率，稀有词更重要 |
| k1 | 词频饱和度参数 |
| b | 长度归一化参数 |

## 下一步学习

- 混合检索（Hybrid Search）
- 重排序（Reranking）